# Coupled 0-D $\leftrightarrow$ 3-D Cluster Dynamics for Irradiated Zirconium

`ZrMicro` (0-D reduced cluster dynamics) coupled to `MoDELib2-NNL` (spatially-resolved
SR-CD / FEM), following **Deliverable D1/M1, *Simulation of Initial Irradiation Damage***
(Po & Ghoniem, 29 July 2026) — [`GW_Phase4_D1M1_NG.pdf`](../../Docs/Formulation/GW_Phase4_D1M1_NG.pdf) —
and the coupling formulation in
[`ZrMicro_MoDELib2_two_time_scale_coupling.tex`](../../Docs/Formulation/ZrMicro_MoDELib2_two_time_scale_coupling.tex).

**Notebook layout**

| Section | Contents |
|---|---|
| 1–4 | Governing equations and numerical methods (reference, no execution) |
| 5 | Environment: paths, directories, and **building both C++ codes if absent** |
| 6 | User controls — one consistent parameter set driving *both* codes |
| 7 | Time-stamped output directory |
| 8 | 0-D reference run + figure suite |
| 9 | Spatial grid and the fast-solve backend |
| 10 | Operator-split dose march, snapshots every `DOSE_INTERVAL` dpa up to `DOSE_MAX` |
| 11 | 3-D field plots: mobile concentrations; immobile densities and contents |
| 12 | Grain-boundary profiles (log density / content vs distance) |
| 13 | Provenance and manifest |

> **Read before interpreting results.** Under the adopted calibration direction the
> resolved model is the reference and `ZrMicro`'s scalar $\delta^{DAD}$ is *derived* from
> `MoDELib`'s tensor $p_m$. That refit has **not** been performed yet, so the 0-D and 3-D
> models presently bias different channels and a 0-D/3-D discrepancy is *expected*
> rather than a coupling defect. Section 6 prints the current status.

## 1. The spatially-resolved (3-D) SR-CD model

*Deliverable §2.* Defect clusters are continuum fields. A limited set of **mobile**
clusters is resolved with fixed defect content; larger **immobile** clusters are
tracked only by number density and total defect content.

### 1.1 Mobile species

`mobileSpeciesVector = -1 1 2 3` selects $\{v,\,i,\,2i,\,3i\}$. Signed sizes $m_k$
carry polarity $p_k=m_k/|m_k|$. `MoDELib` solves the **steady-state** reaction–diffusion PDE
for $\mathbf c_M=[c_v,c_i,c_{2i},c_{3i}]^T$ in fractional (per-atom) units,

$$
\mathbf 0=-\nabla\cdot\mathbf J+\mathbf P_M+\tfrac12\,\mathbf c_M^{T}\mathbf R_2\,\mathbf c_M+(\mathbf E-\mathbf D)\,\mathbf c_M+\mathbf S_{vL}.
$$

**Transport.** $\mathbf J_x=-\mathbf D^x\nabla c_x$ with an anisotropic tensor

$$
D^x_{ij}=D^x_{0,ij}\exp\!\big(-E^{xm}_{ij}/k_BT\big),
$$

from `mobileSpeciesD0_SI` and `mobileSpeciesEnergyMigration_eV`.

**Production.** $\mathbf P_M=G\,\varepsilon\,\mathbf f$ — dose rate, cascade surviving
efficiency, and the partition vector; component-wise this reproduces the 0-D sources
$\{G_v,G_i,\tfrac12 G_{2i},\tfrac13 G_{3i}\}$.

**Second-order reactions.** Pairwise constants use the **sum** of reactant jump frequencies,

$$
\beta_{m,n}=\omega_m+\omega_n,
$$

with the recombination-volume factor $\beta_r$ on the $i$–$v$ channel and combinatorial
factors $2,4$ on like-species reactions. Capture radii follow the point-defect / prismatic-loop geometry

$$
r_m=\begin{cases}\left(\dfrac{3\Omega_0}{4\pi}\right)^{1/3}, & |m_m|=1,\\[2ex]
\left(\dfrac{|m_m|\Omega_0}{\pi b}\right)^{1/2}, & |m_m|>1.\end{cases}
$$

**Thermal emission.** $\alpha_{m,n}=\beta_{m,n}\exp\!\big(-E^b_{m,n}/k_BT\big)$.

**Sinks.** $\mathbf D=\operatorname{diag}\{\boldsymbol\omega\odot\mathbf C_s\}+\mathbf D_d\mathbf K_L$,
splitting a *network* part (identical to 0-D) from a *loop-absorption* part $\mathbf K_L$.
Loop sink strengths sum over **three** immobile families $\mathcal I\mathcal L\in\{c\mathcal L,a\mathcal L,bp\}$:

$$
\langle k^2_{a\mathcal L,m}\rangle=Z^m_{a\mathcal L,\rm tot}\,2\pi r_{a\mathcal L}n_{a\mathcal L},
\qquad
\langle k^2_{c\mathcal L,m}\rangle=Z^m_{c\mathcal L,\rm tot}\,2\pi r_{c\mathcal L}n_{c\mathcal L},
\qquad
\langle k^2_{bp,m}\rangle=\alpha_{bp}s_{bp}\pi r_{bp}n_{bp}.
$$

Note the bi-pyramid law is **volumetric**, not planar.

**DAD bias — generated, not posited.** The diffusion-anisotropy difference follows from
the diffusion tensors themselves,

$$
p_m=\Big(\frac{D^m_c}{D^m_a}\Big)^{1/6},\qquad
Z^{\mathrm{DAD},0}_{a\mathcal L,m}=Z^0_m\frac{p_m+p_m^{-2}}{2},\qquad
Z^{\mathrm{DAD},0}_{c\mathcal L,m}=Z^0_m\,p_m .
$$

**Boundary condition.** Dirichlet (perfect sink),
$c_M(\mathbf x)=c_{\rm bnd,eq}(\mathbf x)$, with

$$
c^k_{\rm bnd,eq}=c^k_0\exp\!\Big(-\frac{m_k t_n\Omega}{k_BT}\Big),
\qquad
c^k_0=|m_k|\exp\!\Big(-\frac{E^f_k+m_k q\,\Omega^r_k}{k_BT}\Big),
$$

$q=-\operatorname{tr}(\boldsymbol\sigma)/3$. A Robin form
$-\mathbf J_k\!\cdot\!\mathbf n=h_k[c_k-c^k_{\rm bnd,eq}]$ recovers the perfect sink as $h_k\to\infty$.

### 1.2 Immobile species (loops)

Four families, vacancy-first: $\mathcal L=\{vL,\,ia_1,\,ia_2,\,ia_3\}$ — one basal
$\langle c\rangle$ vacancy family and three prismatic $\langle a\rangle$ interstitial
variants. Each carries a **density** $n_k$ and a **content** $c_k$ (both per host atom),

$$
\bar m_k=\frac{c_k}{n_k},\qquad r_k=\ell_k\sqrt{\frac{c_k}{n_k}},\qquad
\ell_{vL}=\ell_c,\;\ell_{ia_k}=\ell_a .
$$

**Density equation** (nucleation, annealing, coalescence):

$$
\frac{d\mathbf n_I}{dt}=\dot{\mathbf N}^{\rm nuc}-\mathbf a_n-\Delta\mathbf N,
\qquad
\dot{\mathbf N}^{\rm nuc}=\mathbf w(\boldsymbol\sigma)\,\Phi_{\rm clus}+\mathbf G_L\oslash\mathbf n^{\rm nuc},
$$

with $\Phi_{\rm clus}=R_{i,3i}+R_{2i,2i}$ and $\Delta N_k=(\nu^k_{LL}\phi^k_{LL}+\nu^k_{LN}\phi^k_{LN})n_k$.

**Content equation** (absorption, cascade deposition, seeding, coalescence):

$$
\frac{d\mathbf c_I}{dt}=\dot{\mathbf G}^c+\mathbf P_G+\mathbf s^{\rm nuc}-\mathbf A_c-\Delta\mathbf C .
$$

Each family grows on its like defect and shrinks on the opposite one,

$$
\dot G^c_{vL}=p_{vL}\pi_{vL}\big[Z^{vL}_v\varphi_v-Z^{vL}_i\varphi_i\,g(r_{vL})\big],
\qquad
\dot G^c_{a_kL}=p_{a_kL}\pi_{a_kL}\big[Z^{a_kL}_i\varphi_i-Z^{a_kL}_v\varphi_v\big],
$$

with fluxes $\varphi_v=\omega_v c_v$, $\varphi_i=\omega_i(c_i+2c_{2i}+3c_{3i})$. Each
$\langle a\rangle$ variant sees SIPA at its **own** resolved stress
$\sigma_{kk}=\hat{\mathbf b}_k\!\cdot\!\boldsymbol\sigma\!\cdot\!\hat{\mathbf b}_k$.

**Stress-induced preferential nucleation (SIPN).** Cascade defects partition through the
*formation-volume tensor*,

$$
\chi^k_{i}=\frac{\exp(\boldsymbol\sigma\!:\!\boldsymbol\Omega_k/k_BT)\,I^k_i}{\sum_p\exp(\boldsymbol\sigma\!:\!\boldsymbol\Omega_p/k_BT)\,I^p_i},
$$

whose small-stress limit is $w_{a_k}=\tfrac13(1+\chi\hat\sigma_{kk})$, reducing to $\tfrac13$ at zero stress.

**Bi-pyramid $\to$ loop transition.** Vacancy clusters begin compact and flatten onto the
basal plane:

$$
R_k=(1-S_k)R^k_{\rm pyr}+S_kR^k_{\rm loop},\qquad
R^k_{\rm pyr}=\Big(\frac{m_k\Omega_0}{\sqrt8}\Big)^{1/3},\quad
R^k_{\rm loop}=\sqrt{\frac{m_k\Omega_0}{\pi b_k}},
$$

$$
S_k(m_k)=\Big[1+\exp\!\big(-(m_k-m^k_0)/w_k\big)\Big]^{-1},\quad
m^k_0=\tfrac12(m_{\min}+m_{\max})-m_s,\quad w_k=w_0(m_{\max}-m_{\min}).
$$

The same $S_k$ interpolates the formation-volume tensor between spherical
$\boldsymbol\Omega_{\rm pyr}=(\mathbf 1/3)V_k$ and planar
$\boldsymbol\Omega_{\rm loop}=(\mathbf b\otimes\mathbf b/\mathbf b\!\cdot\!\mathbf b)V_k$ limits.

**Vacancy-loop thermal emission** (a channel the 0-D model omits), evaluated at the mean size:

$$
\alpha_{v,vL}(\bar m)=\beta_{v,vL}\exp\!\big(-E^{v}_b(\bar m)/k_BT\big),
$$

reducing content without removing loops. Released vacancies return to the mobile pool via
$\mathbf S_{vL}$, so the exchange is a pure transfer and the point-defect budget stays closed.

## 2. The reduced 0-D model (`ZrMicro`)

*Deliverable §3.* The integrated state is **19** ODEs,

$$
\mathbf y=\big[\underbrace{C_v,C_i,C_{2i},C_{3i},C_{iL},C_{aiL},C_{vL},C_{avL},C^i_{iL},C^i_{aiL},C^v_{vL},C^v_{avL}}_{12\ \text{physical}},\ \underbrace{\Sigma^{\rm prod}_i\ldots\Sigma^{\rm sink}_v}_{6\ \text{accumulators}},\ \rho_N\big]^T .
$$

**Four** jump frequencies (small clusters get their own migration energy):

$$
\omega_i=z_c\nu_ie^{-E^m_i/k_BT},\quad
\omega_{2i}=z_c\nu_ie^{-E^m_{2i}/k_BT},\quad
\omega_{3i}=\omega_{2i},\quad
\omega_v=z_c\nu_ve^{-E^m_v/k_BT},
$$

with $z_c=48$. Reaction rates use the **sum** of frequencies,
$R_{a,b}=(\omega_a+\omega_b)C_aC_b$, with $\beta_r$ (`recom`) on recombination.

**DAD bias (phenomenological in 0-D).**

$$
Z^a_i=1+\delta_i,\quad Z^a_v=1-\delta_v,\quad Z^c_i=1-\delta_i,\quad Z^c_v=1+\delta_v,
$$

$\delta_i(T)$ linearly interpolated between 300 K and 600 K anchors. Note this forces
$Z^a_m+Z^c_m=2$, a normalization the resolved factors do **not** share.

**Network-only sinks** (loops are handled by the explicit mass-conserving coupling, so
they must not appear here or absorption is double-counted):

$$
C^v_s=\frac{a^2}{z_c}\rho_N,\qquad C^i_s=\frac{a^2}{z_c}Z_N\rho_N .
$$

**Cascade partition** — interstitial and vacancy atom budgets each close to $G$:

$$
G_i+G_{2i}+G_{3i}+G_{iL}+G_{aiL}=G,\qquad G_v+G_{vL}+G_{avL}=G .
$$

**Governing RHS** (12 physical species), e.g.

$$
\frac{dC_v}{dt}=G_v+\frac{c^v_{vL}}{\tau_{vL}}+\frac{c^v_{avL}}{\tau_{avL}}
-\big(R_{i,v}+R_{2i,v}+R_{3i,v}+R_{v,s}+\text{loop\_abs}_v\big),
$$

$$
\frac{dC_i}{dt}=G_i+R_{2i,v}+2E_{2i}+3E_{3i}-\big(R_{i,v}+R_{i,i}+R_{i,2i}+R_{i,3i}+R_{i,s}+\text{loop\_abs}_i\big),
$$

plus four loop-density and four loop-content equations.

**Geometric coalescence** (Avrami overlap), with rate scales set by the *absorbed-flux* climb speed:

$$
\phi_{LL}=1-e^{-\kappa_{LL}\frac43\pi r^3_{LL}N},\quad
\phi_{LN}=1-e^{-\kappa_{LN}\pi r^2_{LN}\rho_N},\quad
\nu_{LL}=c_{LL}v_{\rm abs}N^{1/3},\quad \nu_{LN}=c_{LN}v_{\rm abs}\sqrt{\rho_N}.
$$

**Evolving network density**

$$
\frac{d\rho_N}{dt}=c_{\rho_N}\frac{2\pi}{\Omega}\sum_k r_k\Delta N_{LN,k}-k^{\rm rec}_{\rho_N}\big(\rho_N-\rho_N(0)\big).
$$

**Observables.** Growth/creep strains and dispersed-barrier hardening:

$$
\varepsilon_a=A_a\big(c^i_{iL}+c^i_{aiL}\big),\qquad
\varepsilon_c=-A_c\big(c^v_{vL}+c^v_{avL}\big),
$$

$$
\Delta\tau_X=\alpha_X\mu b\sqrt{N_Xd_X},\quad
\Delta\tau_{SR}=\sqrt{\textstyle\sum_X\Delta\tau_X^2},\quad
\Delta\sigma_y=(\Delta\tau_{SR}+\Delta\tau_{LR})/M .
$$

## 3. Correspondence between the 3-D and 0-D models

*Deliverable §4.* Each 0-D concentration is the volume average of the 3-D field,

$$
C_x\;\longleftrightarrow\;\langle c_x\rangle=\frac1V\int_V c_x(\mathbf x)\,dV .
$$

The structural difference is **transport**: the divergence integrates to the net boundary
flux, so

$$
-\frac1V\int_V\nabla\cdot\mathbf J_x\,dV\;\longleftrightarrow\;-R_{x,s}-\text{loop\_abs}_x .
$$

| Term | 3-D | 0-D | Relation |
|---|---|---|---|
| Mobile concentrations | fields $\mathbf c_M$ | scalars $C_x$ | volume average |
| Transport | $-\nabla\cdot\mathbf J$ | lumped sinks | **interpretation** |
| Production | $\mathbf P_M=G\varepsilon\mathbf f$ | $\{G_v,G_i,\tfrac12G_{2i},\tfrac13G_{3i}\}$ | identical |
| Second-order reactions | $\tfrac12\mathbf c_M^T\mathbf R_2\mathbf c_M$ | $(\omega_a+\omega_b)C_aC_b$ | identical |
| Thermal emission | $\mathbf E\mathbf c_M$ | $E_{2i},E_{3i}$ | identical |
| Network sinks | $\operatorname{diag}\{\boldsymbol\omega\odot\mathbf C_s\}$ | $\omega_xC_xC^x_s$ | identical |
| Point-defect fluxes | $\varphi_v,\varphi_i$ | $\varphi_v,\varphi_i$ | identical |
| Loop absorption | $\mathbf D_d\mathbf K_L$ | mass-conserving `loop_abs` | equivalent |
| DAD bias | $p_m,\,Z^{\mathrm{DAD},0}$ | $\delta_i,\delta_v$ | **interpretation** |
| Boundary condition | Dirichlet $c_{\rm bnd,eq}$ | lumped sink, $C^{\rm eq}_v$ | closure |
| Loop radii | resolved geometry | $r=\ell\sqrt{c/n}$ | closure |

**Three terms do not map exactly**, and each admits a nonzero residual that is physics,
not a bug: the DAD parameterization; the bi-pyramid family (absent in 0-D); and
vacancy-loop thermal emission (present only in 3-D, biting hardest at high $T$ and dose).

## 4. Numerical solution methods

### 4.1 Scale separation and the operator split

Mobile species equilibrate on $\sim\!10^{-3}$–$10^{0}$ s; the loop population evolves over
dpa (i.e. $10^{5}$–$10^{8}$ s). The mobile manifold is therefore treated as **quasi-steady**
(Tikhonov–Fenichel / QSSA), giving the adopted **operator-split QSSA** scheme. Over each
dose step $[\gamma_n,\gamma_{n+1}]$:

1. **Fast solve** — steady mobile field $\mathbf C_M^\star(\mathbf x)$ from the FEM
   reaction–diffusion BVP, with the immobile state frozen.
2. **Slow march** — at every quadrature point independently, integrate the immobile ODEs
   with the mobile species *frozen* at the local $\mathbf C_M^\star$ (`freeze_mobile=1`).
3. Repeat. Splitting error is first order in $\Delta\gamma$ unless Strang-symmetrized.

Because step 2 is **pointwise decoupled**, it maps directly onto `ZrMicro`'s existing
OpenMP batch mode — one case per quadrature point, in a single subprocess.

### 4.2 0-D integrator

SUNDIALS **CVODE**, BDF with a dense linear solver (stiff system, $N=19$).
Benchmarked max relative error $3\times10^{-4}$ at `rtol=1e-6`, `atol=1e-20`.
`ARKODE` tables are available but perform poorly here in pure implicit mode.

| Config | Backend | LinSol | Max rel. error |
|---|---|---|---|
| **A (default)** | CVODE BDF | dense | $3.0\times10^{-4}$ |
| B | CVODE BDF | band | $3.0\times10^{-4}$ |
| C | CVODE BDF | gmres | $1.7\times10^{-2}$ |
| G | ARKODE SDIRK_5_3_4 | dense | $3.0\times10^{-1}$ |

### 4.3 3-D fast solve

Galerkin FEM on a tetrahedral mesh. The steady nonlinear residual
$\mathcal R_M(\mathbf C_M)=\mathbf 0$ is solved by Newton iteration,

$$
\mathbf K_{MM}(\mathbf C_M^{(j)})\,\delta\mathbf C_M=-\mathcal R_M(\mathbf C_M^{(j)}),
\qquad
\frac{\partial\mathcal R^\alpha_M}{\partial C^\delta_M}
=-\nabla\!\cdot\!\Big(\frac{\partial\mathbf J^\alpha_M}{\partial C^\delta_M}\Big)
+R_2^{\alpha\delta\gamma}C^\gamma_M+R_1^{\alpha\delta},
$$

warm-started from the previous dose step. Sparse solves use CHOLMOD/UMFPACK when
available, else Eigen `SimplicialLLT`/`SparseLU`.

### 4.4 Dose as the march variable

Dose accumulates linearly, $\gamma=Gt$, so a dose step maps to
$\Delta t=\Delta\gamma/G$. Snapshots are written every `DOSE_INTERVAL` dpa up to
`DOSE_MAX` (Section 6).

## 5. Environment: paths, directories, and C++ builds

Locates both codebases, verifies the Python environment, and **builds whatever is
missing** — so this notebook is runnable on a fresh machine. Set `FORCE_REBUILD = True`
to rebuild even when binaries already exist.

In [ ]:
# ── Bootstrap: resolve the DisloCluster layout, import the 0-D modules ──────
import os, sys, json, shutil, subprocess, platform, datetime, textwrap
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Every repository location comes from py_utils/paths.py, which finds the root
# by walking up to the `.dislocluster_root` marker file. Nothing below is
# hard-coded, so the repository can be moved or cloned anywhere. Only the
# bootstrap of sys.path is done from the notebook's own location.
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

# Tolerate being launched from ZrMicro/code/ (expected), ZrMicro/, ZrClusterDynamics/ or
# the DisloCluster root.
for _cand in (NOTEBOOK_DIR.parent, NOTEBOOK_DIR, NOTEBOOK_DIR / "ZrMicro",
              NOTEBOOK_DIR / "ZrClusterDynamics" / "ZrMicro"):
    if (_cand / "py_utils" / "paths.py").is_file():
        _ZRMICRO = _cand
        break
else:
    raise RuntimeError(f"Cannot locate ZrMicro/py_utils/paths.py from {NOTEBOOK_DIR}")

if str(_ZRMICRO) not in sys.path:
    sys.path.insert(0, str(_ZRMICRO))

from py_utils import paths

REPO_ROOT  = paths.REPO_ROOT       # DisloCluster/
ZR_ROOT    = paths.ZR_ROOT         # DisloCluster/ZrClusterDynamics/      (0-D code + Docs)
BASE_DIR   = paths.ZRMICRO_DIR     # DisloCluster/ZrClusterDynamics/ZrMicro/
INPUT_DIR  = paths.INPUT_DIR
OUTPUT_DIR = paths.OUTPUT_DIR
CPP_UTILS  = paths.CPP_UTILS
BUILD_DIR  = paths.BUILD_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from py_utils.input_data     import InputData
from py_utils.reaction_rates import ReactionRates
from py_utils.rate_equations import RateEquations
from py_utils.cpp_bridge     import (collect_solver_args, run_cpp_solver,
                                     _find_solver_exe)
from py_utils.post_process   import calculate_derived_quantities
from py_utils.visualization  import plot_results, _get_git_hash
from py_utils.modelib_coupling import pack_y0, run_immobile_step, IMMOBILE_NAMES
from py_utils import modelib_export
from py_utils.modelib_fem    import MoDELibFEMSolver, seed_sim_dir_from_tutorial

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 11})

print("DisloCluster layout")
print("-" * 78)
print(paths.describe())
print("-" * 78)
print(f"python       : {sys.version.split()[0]}  ({sys.executable})")
print(f"numpy        : {np.__version__}   matplotlib: {matplotlib.__version__}")
if paths.venv_python() and Path(sys.executable).resolve() != paths.venv_python().resolve():
    print(f"NOTE: not running the repository interpreter "
          f"({paths.venv_python()}) -- results may not be reproducible.")


# ── Scripted overrides (optional) ───────────────────────────────────────────
# DISLOCLUSTER_NB_CONFIG holds a JSON object that is merged over the control
# blocks below, so this notebook can be executed non-interactively for a
# different domain without editing them:
#
#   {"tag": "hex400", "geometry": {...}, "mesh": {...}, "run": {...},
#    "evl_dir": "<path to an evl/ directory>"}
#
# Unset (the normal interactive case) it is an empty dict and nothing changes.
NB_CFG = json.loads(os.environ.get("DISLOCLUSTER_NB_CONFIG", "{}"))
if NB_CFG:
    print(f"scripted overrides: {sorted(NB_CFG)}")


In [ ]:
# ── Build controls ──────────────────────────────────────────────────────────
FORCE_REBUILD  = False          # rebuild both C++ codes even if binaries exist
BUILD_TYPE     = "Release"      # Release recommended (batch mode is called often)

# 3-D code (MoDELib). Resolved from the repository layout by py_utils/paths.py;
# override with the MODELIB_ROOT / MODELIB_BUILD environment variables if the
# checkout lives outside DisloCluster/.
MODELIB_ROOT   = paths.MODELIB_ROOT
MODELIB_BUILD  = paths.MODELIB_BUILD
MODELIB_BUILD_SCRIPT = paths.MODELIB_BUILD_SCRIPT

# MoDELib is built inside WSL on Windows; its DDomp is a Linux ELF, so it is
# invoked through wsl.exe. On Linux/macOS the native binary is used directly.
USE_WSL = paths.use_wsl()

# d:/x/y -> /mnt/d/x/y  (shared with modelib_fem so both agree on the mapping)
windows_to_wsl = paths.windows_to_wsl


def _run(cmd, cwd=None, timeout=3600):
    """Run a command, echoing a trimmed transcript. Returns (rc, stdout+stderr)."""
    print(f"  $ {' '.join(str(c) for c in cmd)}")
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd),
                       capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or "") + (p.stderr or "")
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    for l in tail:
        print("    " + l[:150])
    return p.returncode, out


print(f"ZrMicro (0-D) : {BASE_DIR}")
print(f"MoDELib (3-D) : {MODELIB_ROOT}")
print(f"build via WSL : {USE_WSL}")


In [ ]:
# ── 1) ZrMicro C++ solver (SUNDIALS CVODE) ──────────────────────────────────
def ensure_zrmicro_solver(force=False):
    exe = _find_solver_exe(BASE_DIR)
    if exe and not force:
        print(f"[ZrMicro] solver found : {exe}")
        return Path(exe)

    print("[ZrMicro] solver.exe not found -- configuring and building")

    # A CMake cache records the absolute paths it was configured with, so moving
    # or renaming the repository makes "cmake -S ... -B ..." hard-error with
    # "the current CMakeCache.txt directory ... is different". Discard a cache
    # that belongs to a different location instead of failing. (The MoDELib
    # build script does the same for the 3-D tree.)
    _cache = BUILD_DIR / "CMakeCache.txt"
    if _cache.is_file():
        _home = next((l.split("=", 1)[1].strip()
                      for l in _cache.read_text(errors="ignore").splitlines()
                      if l.startswith("CMAKE_HOME_DIRECTORY:")), "")
        if _home and Path(_home) != Path(CPP_UTILS):
            print(f"  discarding stale cache configured for {_home}")
            shutil.rmtree(BUILD_DIR, ignore_errors=True)

    rc, _ = _run(["cmake", "-S", CPP_UTILS, "-B", BUILD_DIR,
                  f"-DCMAKE_BUILD_TYPE={BUILD_TYPE}"])
    if rc != 0:
        raise RuntimeError("CMake configure failed for ZrMicro (see log above)")
    rc, _ = _run(["cmake", "--build", BUILD_DIR, "--config", BUILD_TYPE])
    if rc != 0:
        raise RuntimeError("CMake build failed for ZrMicro (see log above)")

    exe = _find_solver_exe(BASE_DIR)
    if not exe:
        raise RuntimeError(
            "Build reported success but solver.exe is still not found. "
            f"Searched under {BUILD_DIR}.")
    print(f"[ZrMicro] built : {exe}")
    return Path(exe)


ZRMICRO_SOLVER = ensure_zrmicro_solver(FORCE_REBUILD)

In [ ]:
# ── 2) MoDELib2-NNL (DDomp + pyMoDELib) ─────────────────────────────────────
# NOTE: exit status is NOT a reliable build signal here -- the build script ends
# with `ls ... || echo WARN`, so it returns 0 even when nothing was produced.
# Artifacts are therefore verified on disk.
def ensure_modelib(force=False):
    femsolver = MoDELibFEMSolver(
        sim_dir=str(OUTPUT_DIR / "modelib_sim"),
        modelib_root=str(MODELIB_ROOT),
        modelib_build_dir=str(MODELIB_BUILD),
        wsl_exec=USE_WSL,
    )
    if femsolver.available() and not force:
        print(f"[MoDELib] DDomp found : {femsolver.find_ddomp()}")
        return femsolver

    if not MODELIB_ROOT.is_dir():
        print(f"[MoDELib] repo NOT found at {MODELIB_ROOT} -- 3-D backend unavailable.")
        print("          Expected inside the repo as DisloCluster/MoDELib3/, or set "
                  "MODELIB_ROOT.")
        return femsolver

    if not MODELIB_BUILD_SCRIPT.is_file():
        print(f"[MoDELib] build script missing: {MODELIB_BUILD_SCRIPT}")
        return femsolver

    print("[MoDELib] binaries absent -- building (this takes 10-30 min)")
    if USE_WSL:
        # Run as root: the script's `sudo apt-get` needs no password as root, and
        # MSYS_NO_PATHCONV stops Git Bash rewriting /mnt/... into a Windows path.
        env = dict(os.environ, MSYS_NO_PATHCONV="1")
        cmd = ["wsl.exe", "-u", "root", "-e", "bash", "-lc",
               f"bash {windows_to_wsl(MODELIB_BUILD_SCRIPT)} "
               f"{windows_to_wsl(MODELIB_ROOT)}"]
        print(f"  $ {' '.join(cmd)}")
        p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=7200)
        for l in [l for l in (p.stdout + p.stderr).splitlines() if l.strip()][-15:]:
            print("    " + l[:150])
    else:
        _run(["bash", MODELIB_BUILD_SCRIPT, MODELIB_ROOT], timeout=7200)

    # Verify on disk rather than trusting the exit code.
    if femsolver.available():
        print(f"[MoDELib] built : {femsolver.find_ddomp()}")
    else:
        print("[MoDELib] BUILD DID NOT PRODUCE DDomp -- 3-D backend stays unavailable.")
        print("          Common causes: WSL first-run (OOBE) not completed; missing")
        print("          libfftw3-dev; CMake >= 4 rejecting cmake_minimum_required(3.1).")
    return femsolver


FEMSOLVER = ensure_modelib(FORCE_REBUILD)
MODELIB_OK = FEMSOLVER.available()
print(f"\nMoDELib 3-D backend available: {MODELIB_OK}")

## 6. User controls

One parameter set drives **both** codes. `DOSE_INTERVAL` ($x$) and `DOSE_MAX` ($y$)
control the snapshot cadence and the end of the march.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  SECTION 6a — GEOMETRY AND MESH
# ════════════════════════════════════════════════════════════════════════════
# The simulation domain and its discretisation. Both are built by Gmsh through
# Gmsh/ (repository root)generate_mesh.py, which writes the msh 2.2 ASCII format
# with tetrahedra only and the region ID in tags[1] -- the only thing MoDELib's
# reader accepts. Meshes are cached under Gmsh/meshes/ on a hash of this spec,
# so re-running with unchanged settings costs nothing.

GEOMETRY = {
    "type":       "cubic",      # "cubic" | "hexagonal"
    # cubic      : side length (scaled per axis by `aspect`)
    # hexagonal  : corner-to-corner diameter of the hexagonal cross-section,
    #              prism axis along z (the c-axis), one vertex on +x so a prism
    #              facet is normal to an <a>-type direction
    "size_nm":    1000.0,
    "height_nm":  None,         # hexagonal only; None -> equal to size_nm
    "aspect":     (1.0, 1.0, 1.0),   # cubic only: multipliers of size_nm
}

MESH = {
    "target_elements":       15000,  # sets the interior element size
    "lc_nm":                 None,   # explicit element size [nm]; overrides above
    "element_order":         1,      # 1 = linear tet (type 4), 2 = quadratic (type 11)

    # Boundary-layer refinement. All of the spatial structure lives within about
    # two depletion lengths (~150 nm) of the surface; a uniform mesh either
    # wastes elements on the interior plateau or under-resolves the denuded zone.
    "boundary_layer":        True,
    "boundary_thickness_nm": 150.0,

    # Layer element size, resolved in this order -- the first one set wins:
    #   boundary_lc_nm         size in nm, stated outright
    #   boundary_layers_across size = thickness / N
    #   boundary_size_ratio    size = interior size x ratio
    # so boundary_size_ratio only takes effect if the two above are None.
    "boundary_lc_nm":        None,
    "boundary_layers_across": 8,
    "boundary_size_ratio":   0.35,

    # Geometric expansion ratio out of the layer: each successive element is
    # this much larger than the last. None -> grade over 2 x the thickness.
    "boundary_expansion":    None,

    "optimize":              True,
    "regenerate":            False,  # force a rebuild even if the cache hits
}


# Scripted overrides from DISLOCLUSTER_NB_CONFIG (empty when interactive).
GEOMETRY.update(NB_CFG.get("geometry", {}))
MESH.update(NB_CFG.get("mesh", {}))


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  SECTION 6b — RUN CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
# Physical conditions and the dose schedule, applied to BOTH codes.
#
# The march is seeded at `dose_seed` from the 0-D reference and then advances in
# `n_intervals` equal steps to `dose_max`. The defaults give
#     1, 6, 11, 16, 21, 26 dpa
# i.e. a 26 dpa run reported at six doses.

RUN = {
    # ---- physical conditions ----
    "temperature_K":   573.0,
    "dose_rate_dpa_s": 1.0e-7,
    "sigma_n_Pa":      0.0,      # applied normal stress; 0 => w_k = 1/3

    # ---- dose schedule ----
    "dose_seed":       0.1,      # must be > 0; the march starts here
    "dose_max":        20.1,
    "n_intervals":     4,

    # ---- integration ----
    "rtol":                  1e-6,
    "atol":                  1e-20,
    "substeps_per_interval": 20,   # immobile substeps inside ONE dose interval

    # ---- 1-D march grid (the pointwise slow march) ----
    "n_points":      25,
    "domain_in_ell": 12.0,       # domain length in depletion lengths
}

# ---- 0-D reference span. Set independently of the march so the 0-D run can be
#      made bit-identical to a previous reference run; it must still cover
#      dose_max.
T_END_0D    = 1.0e9      # [s] -> 100 dpa at G = 1e-7
N_POINTS_0D = 200

# ---- runtime flags ----
RUN_0D_REFERENCE   = True
RUN_3D_MARCH       = True
FAST_SOLVE_BACKEND = "auto"     # "auto" | "modelib" | "placeholder"
SHOW_INLINE        = True

# ---- 3-D figure controls (Figs 19-27, one quantity per file) ----
FIG3D_OVERLAYS = True     # per-family platelet overlays (Figs 22-23); the slow part
FIG3D_N_LOOPS  = 26       # platelets drawn per overlay panel (decorative only)
FIG3D_PLANE    = "y"      # mid-cut for the 3d/ panels: "x" | "y" | "z".
                          # "y" suits the cube, where every cut is equivalent by
                          # symmetry. Use "z" for a hexagonal prism: it cuts
                          # normal to the c-axis and shows the hexagonal section
                          # with the depletion rim on all six prism faces, where
                          # a "y" cut runs through two opposite vertices and is
                          # just a rectangle.
GB_MAX_NM      = 150.0    # extent of the gb/ profiles
GB_N_BINS      = 60

# ════════════════════════════════════════════════════════════════════════════
# Derived: flat aliases kept so the rest of the notebook reads unchanged.
TEMPERATURE_K = RUN["temperature_K"]
DOSE_RATE     = RUN["dose_rate_dpa_s"]
SIGMA_N_PA    = RUN["sigma_n_Pa"]
DOSE_START    = RUN["dose_seed"]
DOSE_MAX      = RUN["dose_max"]
N_POINTS      = RUN["n_points"]
DOMAIN_IN_ELL = RUN["domain_in_ell"]

DOSE_SNAPSHOTS = np.linspace(RUN["dose_seed"], RUN["dose_max"],
                             RUN["n_intervals"] + 1)
DOSE_INTERVAL  = float(DOSE_SNAPSHOTS[1] - DOSE_SNAPSHOTS[0])

SOLVER_CONFIG = {
    "t_begin":  1e-1,
    "t_end":    T_END_0D,
    "n_points": N_POINTS_0D,
    "rtol":     RUN["rtol"],
    "atol":     RUN["atol"],
    "log_time": True,
    "solver_method": {"backend": "cvode", "lmm": "bdf", "linsol": "dense"},
}

assert DOSE_RATE * T_END_0D >= DOSE_MAX - 1e-12, (
    f"0-D run reaches {DOSE_RATE*T_END_0D:g} dpa but dose_max = {DOSE_MAX:g} dpa")
assert RUN["dose_seed"] > 0, "dose_seed must be > 0"

print(f"T = {TEMPERATURE_K} K   G = {DOSE_RATE:.3e} dpa/s   "
      f"sigma = {SIGMA_N_PA:.3e} Pa")
print(f"dose schedule : {DOSE_START:g} -> {DOSE_MAX:g} dpa in "
      f"{RUN['n_intervals']} intervals of {DOSE_INTERVAL:g} dpa")
print(f"snapshots     : {np.round(DOSE_SNAPSHOTS, 4)}")
print(f"0-D span      : t_end = {T_END_0D:.3e} s = {DOSE_RATE*T_END_0D:g} dpa "
      f"in {N_POINTS_0D} points")
print(f"tolerances    : rtol = {RUN['rtol']:.1e}   atol = {RUN['atol']:.1e}")


# Scripted overrides from DISLOCLUSTER_NB_CONFIG (empty when interactive).
RUN.update(NB_CFG.get("run", {}))

# The figure controls are plain names rather than dict members, so they are
# merged one by one from the "figures" section of DISLOCLUSTER_NB_CONFIG.
_fig = NB_CFG.get("figures", {})
FIG3D_OVERLAYS = _fig.get("overlays", FIG3D_OVERLAYS)
FIG3D_N_LOOPS = _fig.get("n_loops", FIG3D_N_LOOPS)
FIG3D_PLANE = _fig.get("plane", FIG3D_PLANE)
GB_MAX_NM = _fig.get("gb_max_nm", GB_MAX_NM)
GB_N_BINS = _fig.get("gb_n_bins", GB_N_BINS)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  SECTION 6c — PARAMETER OVERRIDES (applied on top of the workbook)
# ════════════════════════════════════════════════════════════════════════════
# ---- Parameter overrides (applied on top of the Excel workbook) ------------
# Keys are routed automatically to whichever sheet already defines them.
#
# REPRODUCIBILITY: this block reproduces the 0-D reference run
#   output/20260622_144021_7959445/
# The input workbook has drifted since that run (28 parameters differ; 11 are
# now absent from it entirely), so every value the reference depended on is
# pinned here rather than inherited from the spreadsheet. Full precision is
# taken from ZrMicro.ipynb's fitted override block -- provenance.md stores only
# %.4e, so re-reading the rounded text would NOT reproduce the run.
#
# `rho` and `rho_600` are deliberately NOT pinned: the workbook holds
# 1.88004e14 / 2.00116e14 while provenance shows the rounded 1.8800e+14 /
# 2.0012e+14 -- the same values, and pinning the rounded text would inject a
# 2e-5 relative error.
# Optional regression baseline. The 0-D reference run below was produced in
# the pre-DisloCluster tree; if its directory is not present in this checkout
# the verification cell is skipped rather than failing. Point this at any
# ZrMicro output/<run> directory (relative to ZrMicro/) to re-enable it.
REFERENCE_RUN = Path("output/20260622_144021_7959445")
if not (paths.ZRMICRO_DIR / REFERENCE_RUN / "provenance.md").is_file():
    REFERENCE_RUN = None

PARAMETER_OVERRIDES = {
    "T"               : TEMPERATURE_K,
    "G"               : DOSE_RATE,
    # ── fitted parameter set of the reference run ──────────────────────────
    "delta_DAD_i_300" : 0.153343,
    "delta_DAD_i_600" : 0.0640797,
    "delta_DAD"       : 0.107886,
    "Z_N"             : 1.01,
    "Z_iL"            : 1.34408,
    "Z_vL"            : 1.02,        # not in ZrMicro.ipynb's block; from provenance
    "Q"               : 0.287931,
    "c_LL_a"          : 120.577,
    "c_LN_a"          : 1130.85,
    "c_LL_c"          : 0.162483,
    "c_LN_c"          : 2.95829,
    "kappa_LL"        : 2.0,         # not in ZrMicro.ipynb's block; from provenance
    "kappa_LN"        : 0.4,         # not in ZrMicro.ipynb's block; from provenance
    "epsilon_iL"      : 0.00211286,
    "epsilon_vL"      : 0.005,
    "epsilon_2i"      : 0.01,
    "epsilon_3i"      : 0.01,
    "n_iL_nuc"        : 300,
    "n_vL_nuc"        : 400,
    "E_m_i"           : 0.759101,
    "E_m_2i"          : 1.36292,
    "E_b_2i"          : 0.957033,
    "E_a_vL"          : 0.470079,
    "tau_vL0"         : 1.05145,
    "c_rhoN"          : 0.001,
    "k_rhoN_rec"      : 3.47726e-05,
}


In [ ]:
# ── Build the mesh ──────────────────────────────────────────────────────────
# generate_mesh lives in Gmsh/ (repository root). It is imported by path rather
# than installed, so the mesh generator stays next to the .msh files it writes.
if str(paths.GMSH_DIR) not in sys.path:
    sys.path.insert(0, str(paths.GMSH_DIR))
import generate_mesh as gm

MESH_SPEC = gm.MeshSpec(
    geometry=GEOMETRY["type"],
    size_nm=GEOMETRY["size_nm"],
    height_nm=GEOMETRY["height_nm"],
    aspect=tuple(GEOMETRY["aspect"]),
    target_elements=MESH["target_elements"],
    lc_nm=MESH["lc_nm"],
    element_order=MESH["element_order"],
    boundary_layer=MESH["boundary_layer"],
    boundary_thickness_nm=MESH["boundary_thickness_nm"],
    boundary_lc_nm=MESH["boundary_lc_nm"],
    boundary_layers_across=MESH["boundary_layers_across"],
    boundary_size_ratio=MESH["boundary_size_ratio"],
    boundary_expansion=MESH["boundary_expansion"],
    optimize=MESH["optimize"],
)

MESH_RESULT = gm.generate(MESH_SPEC, force=MESH["regenerate"])
_lx, _ly, _lz = MESH_RESULT.extents_nm
_lc = MESH_SPEC.interior_lc_nm()

print(f"geometry     : {GEOMETRY['type']}")
print(f"extents      : {_lx:.1f} x {_ly:.1f} x {_lz:.1f} nm")
print(f"element size : {_lc:.2f} nm interior", end="")
if MESH["boundary_layer"]:
    # Read the layer size back off the spec: which of the three controls above
    # actually applies depends on which are set, so recomputing it here would
    # not necessarily agree with the mesh that was built.
    _lb = MESH_SPEC.layer_lc_nm()
    print(f"   {_lb:.2f} nm within {MESH['boundary_thickness_nm']:g} nm of the "
          f"surface ({MESH['boundary_thickness_nm']/_lb:.1f} across)")
else:
    print("   (uniform)")
print(f"nodes / tets : {MESH_RESULT.n_nodes} / {MESH_RESULT.n_tets}  "
      f"(order {MESH['element_order']})")
print(f"mesh file    : {MESH_RESULT.path}")
print()
print(gm.polycrystal_F_block(MESH_SPEC))


In [ ]:
# ── Load input data and build the 0-D model chain ───────────────────────────
INPUT_FILE = INPUT_DIR / "input_parameters.xlsx"
if not INPUT_FILE.exists() and (INPUT_DIR / "Zr_input_parameters.xlsx").exists():
    INPUT_FILE = INPUT_DIR / "Zr_input_parameters.xlsx"
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"No input workbook found in {INPUT_DIR}")
print(f"input workbook : {INPUT_FILE.name}")

import types

idata = InputData(str(INPUT_FILE))

# Apply overrides, routing each key to the sheet that already defines it.
_sheets = {"Material_Environment": idata.material_params,
           "Physical_Properties":  idata.physical_props,
           "Model_Parameters":     idata.model_params}
_applied = []
for k, v in PARAMETER_OVERRIDES.items():
    if v is None:
        continue
    tgt = next((n for n, d in _sheets.items() if k in d), None)
    if tgt is None:
        idata.model_params[k] = v
        _applied.append((k, "Model_Parameters (new)", None, v))
    else:
        _applied.append((k, tgt, _sheets[tgt][k], v))
        _sheets[tgt][k] = v
idata.calculate_derived_parameters()

rr = ReactionRates(idata)
re = RateEquations(idata, rr)
sim = types.SimpleNamespace(input_data=idata, reaction_rates=rr,
                            rate_equations=re, input_file=str(INPUT_FILE))

G     = float(idata.material_params["G"])
T_K   = float(idata.material_params["T"])
Omega = float(idata.physical_props["Omega"])
A_a   = float(idata.model_params["A_a"])
A_c   = float(idata.model_params["A_c"])

if _applied:
    print(f"\n{'parameter':<14}{'sheet':<26}{'workbook':>14}  ->  override")
    print("-" * 74)
    for k, s, old, new in _applied:
        o = "n/a" if old is None else (f"{old:.4e}" if isinstance(old, (int, float)) else str(old))
        print(f"{k:<14}{s:<26}{o:>14}  ->  {new}")

print(f"\nT = {T_K} K    G = {G:.3e} dpa/s    Omega = {Omega:.3e} m^3")
base_cli = collect_solver_args(sim, SOLVER_CONFIG)
print(f"base_cli assembled: {len(base_cli)} parameters for the C++ solver")

In [ ]:
# ── 0-D <-> 3-D consistency check ───────────────────────────────────────────
# Compares the ZrMicro conditions against MoDELib's material file and reports
# where the two models are known NOT to correspond (deliverable Sec. 4.3).
ZR_MATERIAL = paths.MODELIB_MATERIAL


def read_modelib_material(path):
    """Parse `key=value;` entries from a MoDELib material file."""
    if not Path(path).is_file():
        return {}
    txt = Path(path).read_text(errors="ignore")
    txt = "\n".join(l.split("#")[0] for l in txt.splitlines())
    out, buf = {}, ""
    for chunk in txt.split(";"):
        if "=" not in chunk:
            continue
        k, _, v = chunk.partition("=")
        out[k.strip()] = " ".join(v.split())
    return out


mat = read_modelib_material(ZR_MATERIAL)
rows = []
if mat:
    g3d = float(mat.get("doseRate_dpaPerSec", "nan"))
    rows.append(("dose rate G [dpa/s]", f"{G:.3e}", f"{g3d:.3e}",
                 "MATCH" if abs(g3d - G) <= 1e-12 + 1e-6 * abs(G) else "MISMATCH"))
    rows.append(("mobileSpeciesVector", "Cv,Ci,C2i,C3i",
                 mat.get("mobileSpeciesVector", "?"), "-"))

    # DAD bias. MoDELib now carries the fitted (Z0_m, p_m) of Eq.(15), calibrated
    # to reproduce ZrMicro's empirical symmetric factors EXACTLY (closed-form
    # inversion: p = (2 Z_a/Z_c - 1)^(-1/3), Z0 = Z_c/p). Compare the resulting
    # capture efficiencies family by family rather than comparing p_m to a delta.
    try:
        p_m = np.array([float(t) for t in mat["dadAnisotropy"].split()])
        Z0 = np.array([float(t) for t in mat["dadZ0"].split()])
        Za3 = Z0 * 0.5 * (p_m + p_m ** -2.0)      # <a> loops, Eq.(15)
        Zc3 = Z0 * p_m                            # <c> loops, Eq.(15)
        tgt = {  # ZrMicro empirical symmetric forms, Eqs.(63)-(64)
            "Z_v^a": (float(idata.derived["Z_v_a"]), Za3[0]),
            "Z_v^c": (float(idata.derived["Z_v_c"]), Zc3[0]),
            "Z_i^a": (float(idata.derived["Z_i_a"]), Za3[1]),
            "Z_i^c": (float(idata.derived["Z_i_c"]), Zc3[1]),
        }
        for nm, (z0d, z3d) in tgt.items():
            rel = abs(z3d - z0d) / max(abs(z0d), 1e-30)
            rows.append((f"{nm} (DAD capture eff.)", f"{z0d:.6f}", f"{z3d:.6f}",
                         "MATCH" if rel < 1e-6 else f"differ {rel:.1e}"))
        # the fit is NOT the anisotropy the migration tensors imply -- show both
        em = np.array([float(t) for t in
                       mat["mobileSpeciesEnergyMigration_eV"].split()]).reshape(-1, 6)
        kT = 8.617333262e-5 * T_K
        p_tensor = np.exp(-(em[:, 5] - em[:, 0]) / (6.0 * kT))
        for nm, pf, pt in zip(("v", "i", "2i", "3i"), p_m, p_tensor):
            rows.append((f"p_{nm}: fitted vs tensors", f"{pt:.4f} (tensors)",
                         f"{pf:.4f} (fitted)",
                         "consistent" if abs(pf - pt) < 1e-3 else "CALIBRATED, not derived"))
    except Exception as e:
        rows.append(("DAD bias", "-", f"parse failed: {e}", "?"))

print(f"{'quantity':<30}{'ZrMicro (0-D)':>22}{'MoDELib (3-D)':>22}   status")
print("-" * 92)
for q, a, b, s in rows:
    print(f"{q:<30}{a:>22}{b:>22}   {s}")
if not rows:
    print(f"(material file not found: {ZR_MATERIAL})")

print(textwrap.dedent(f"""
    ------------------------------------------------------------------
    DAD bias: RECONCILED. (Z0_m, p_m) in Zr3d_ghoniem.txt (the coupled
      material, resolved by paths.MODELIB_MATERIAL) are fitted by closed-form
      inversion of Eq.(15) against the 0-D symmetric forms Eqs.(63)-(64), exactly
      (residual ~1e-16), so both models now apply the SAME capture efficiencies
      to the SAME channels -- see the MATCH rows above. A residual in eps_a/eps_c
      is therefore a genuine defect, not expected bookkeeping.
      COST: the fitted p_m is NOT the anisotropy the migration tensors imply, so
      these runs do not exercise the "DAD generated from the tensorial flux"
      mechanism of deliverable Sec. 2 -- Eq.(15) is acting as a 2-parameter
      fitting form. The fit is also tied to T={T_K:g} K (delta_i is
      T-interpolated in 0-D); refit dadAnisotropy/dadZ0 if T changes.
      NOTE: Zr4.txt is the standalone 3-D calibration and carries NO dad*
      keys -- checking against it reports a parse failure, not a match.

    REMAINING NON-CORRESPONDENCES (deliverable Sec. 4.3) -- expected, not bugs:
      1. Bi-pyramid : the sub-critical vacancy family exists only in 3-D.
      2. Loop emission: size-dependent vacancy emission exists only in 3-D
                      (0-D uses a constant ~5 eV surrogate).
      3. Nucleation : the 3-D implementation is cascade-only; homogeneous
                      clustering (Phi_clus) and the SIPN stress weights are not
                      yet wired, so it is exact only at zero stress.
    ------------------------------------------------------------------"""))

## 7. Time-stamped output directory

`output/<YYYYMMDD_HHMMSS>_<git-hash>_coupled0d3d/` with `0d/`, `3d/`, and `gb/`
subfolders. A numeric suffix is appended if the name already exists, so a run
never overwrites a previous one.

In [ ]:
# ── Create the unique, time-stamped run directory ───────────────────────────
def make_run_dir(base, tag="coupled0d3d"):
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    ghash = paths.git_hash()          # DisloCluster root, else the sub-checkouts
    root = Path(base) / f"{stamp}_{ghash}_{tag}"
    n = 1
    while root.exists():                       # guarantee uniqueness
        root = Path(base) / f"{stamp}_{ghash}_{tag}_{n}"
        n += 1
    for sub in ("0d", "3d", "gb"):
        (root / sub).mkdir(parents=True, exist_ok=True)
    return root


RUN_DIR = make_run_dir(OUTPUT_DIR, tag=NB_CFG.get("tag", "coupled0d3d"))
DIR_0D, DIR_3D, DIR_GB = RUN_DIR / "0d", RUN_DIR / "3d", RUN_DIR / "gb"
print(f"run directory : {RUN_DIR}")


def savefig(fig, path, name):
    """Save a figure as PNG and optionally display it inline."""
    p = Path(path) / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    print(f"  saved {p.relative_to(RUN_DIR)}")
    if SHOW_INLINE:
        plt.show()
    else:
        plt.close(fig)
    return p

In [ ]:
# ── Write the configured MoDELib simulation directory ───────────────────────
# The geometry and run controls above are materialised into a self-contained
# sim dir under the run directory: the generated mesh, a polycrystal.txt whose F
# restores the physical size from the unit mesh, and a DD.txt whose dose
# schedule matches DOSE_SNAPSHOTS. The tutorial case is left untouched.
#
# The solve is NOT launched here -- it takes hours. The command to run it is
# printed at the end of this cell; the figures below read whichever evl/ output
# EVL_DIR points at.
import re

SIM_CFG_DIR = RUN_DIR / "sim"
(SIM_CFG_DIR / "inputFiles").mkdir(parents=True, exist_ok=True)
(SIM_CFG_DIR / "evl").mkdir(exist_ok=True)
(SIM_CFG_DIR / "F").mkdir(exist_ok=True)

# Start from the verification case's inputFiles, then override what we control.
for _f in (paths.COUPLED_SIM_TUTORIAL / "inputFiles").glob("*"):
    if _f.is_file() and _f.suffix != ".msh":
        shutil.copy2(_f, SIM_CFG_DIR / "inputFiles" / _f.name)
shutil.copy2(MESH_RESULT.path, SIM_CFG_DIR / "inputFiles" / "mesh.msh")

# polycrystal.txt: mesh file, F, and the temperature
_pc = SIM_CFG_DIR / "inputFiles" / "polycrystal.txt"
_t = _pc.read_text()
_t = re.sub(r"meshFile=.*?;", "meshFile=mesh.msh;", _t)
_t = re.sub(r"F=.*?;", gm.polycrystal_F_block(MESH_SPEC), _t, flags=re.S)
_t = re.sub(r"absoluteTemperature=[^;]*;",
            f"absoluteTemperature={RUN['temperature_K']:g};", _t)
_pc.write_text(_t)

# DD.txt: one output step per dpa, enough steps to reach dose_max
_dd = SIM_CFG_DIR / "inputFiles" / "DD.txt"
_t = _dd.read_text()
DOSE_PER_STEP = 1.0
_nsteps = int(round(RUN["dose_max"] / DOSE_PER_STEP))
_t = re.sub(r"Nsteps=\d+", f"Nsteps={_nsteps}", _t)
_t = re.sub(r"outputFrequency=\d+", "outputFrequency=1", _t)
_dd.write_text(_t)

print(f"sim dir      : {SIM_CFG_DIR}")
print(f"mesh         : inputFiles/mesh.msh  ({MESH_RESULT.n_nodes} nodes)")
print(f"temperature  : {RUN['temperature_K']:g} K")
print(f"dose steps   : Nsteps={_nsteps} of {DOSE_PER_STEP:g} dpa "
      f"-> {RUN['dose_max']:g} dpa")
print()
print("To run the 3-D solve (hours):")
_sim_wsl = paths.windows_to_wsl(SIM_CFG_DIR) if paths.use_wsl() else SIM_CFG_DIR
_bin = paths.windows_to_wsl(paths.MODELIB_BUILD / "tools") if paths.use_wsl() \
       else paths.MODELIB_BUILD / "tools"
print(f"  wsl -e bash -lc \"cd {_sim_wsl} && "
      f"{_bin}/MicrostructureGenerator/microstructureGenerator . && "
      f"{_bin}/DDomp/DDomp .\"")

# Which evl/ the figures below are rendered from. Point this at SIM_CFG_DIR
# once the solve above has been run.
EVL_DIR = (Path(NB_CFG["evl_dir"]) if NB_CFG.get("evl_dir")
           else paths.COUPLED_SIM_TUTORIAL / "evl")
print()
print(f"figures read : {EVL_DIR}")
if not (EVL_DIR / "cdNodes.txt").is_file():
    print("  WARNING: no cdNodes.txt there -- the 3d/ and gb/ cells will be skipped.")


## 8. 0-D reference run

Supplies (a) the well-mixed regression reference, and (b) the bulk mobile field the
spatial march depletes near the boundary. The standard `ZrMicro` figure suite is written
to `0d/`.

In [ ]:
# ── Run the 0-D reference ───────────────────────────────────────────────────
ref = None
if RUN_0D_REFERENCE:
    print("Running 0-D reference (CVODE/BDF) ...")
    ref = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)
    if ref is None:
        raise RuntimeError("0-D reference run failed -- check the solver build.")

    ref_dose = G * np.asarray(ref["time"])
    Cref = ref["concentrations"]
    MOB = ("Cv", "Ci", "C2i", "C3i")
    IMM = ("CiL", "CaiL", "CvL", "CavL", "CiL_i", "CaiL_i", "CvL_v", "CavL_v")

    def refmob(d): return np.array([np.interp(d, ref_dose, Cref[n]) for n in MOB])
    def refimm(d): return np.array([np.interp(d, ref_dose, Cref[n]) for n in IMM])
    def refrho(d): return float(np.interp(d, ref_dose, ref["rho_N"]))

    def eps_from_y(y):    return A_a * (y[8] + y[9]), -A_c * (y[10] + y[11])
    def eps_from_ref(d):
        im = refimm(d)
        return A_a * (im[4] + im[5]), -A_c * (im[6] + im[7])

    print(f"  reached {ref_dose[-1]:.3f} dpa in {len(ref_dose)} points")
    print(f"  at {DOSE_MAX:g} dpa :  eps_a = {eps_from_ref(DOSE_MAX)[0]:.4e}   "
          f"eps_c = {eps_from_ref(DOSE_MAX)[1]:.4e}")
    print(f"                     Cv = {refmob(DOSE_MAX)[0]:.4e}   "
          f"Ci = {refmob(DOSE_MAX)[1]:.4e}")
else:
    print("RUN_0D_REFERENCE = False -- skipping.")

In [ ]:
# ── Verify the 0-D run against the reference ────────────────────────────────
# provenance.md stores %.4e, so agreement is only checkable to ~5 significant
# figures. A relative difference at the 1e-5 level therefore means "identical
# to the precision the reference recorded", not a genuine mismatch.
REF_DIR = (BASE_DIR / REFERENCE_RUN) if REFERENCE_RUN else None
TOL_REL = 2e-4          # generous w.r.t. the 5-sig-fig reference text


def parse_provenance(path):
    import re
    txt = Path(path).read_text(encoding="utf-8")
    out = {}
    for sec in ("Final Concentrations", "Final Mechanical Properties",
                "Derived Parameters"):
        m = re.search(rf"^## {re.escape(sec)}\s*$(.*?)(?=^## |\Z)", txt, re.S | re.M)
        if not m:
            continue
        for line in m.group(1).splitlines():
            if not line.startswith("|"):
                continue
            p = [c.strip() for c in line.strip("|").split("|")]
            if len(p) != 2 or p[0] in ("Parameter", "") or set(p[1]) <= {"-"}:
                continue
            try:
                out[p[0]] = float(p[1])
            except ValueError:
                pass
    return out


if ref is not None and REF_DIR is not None and (REF_DIR / "provenance.md").is_file():
    target = parse_provenance(REF_DIR / "provenance.md")
    got = {n: float(Cref[n][-1]) for n in Cref}
    got["strain_a"]  = float(ref["mechanical_properties"]["strain_a"][-1])
    got["strain_c"]  = float(ref["mechanical_properties"]["strain_c"][-1])
    got["hardening"] = float(ref["mechanical_properties"]["hardening"][-1])
    for k in ("Z_i_a", "Z_v_a", "Z_i_c", "Z_v_c"):
        if k in idata.derived:
            got[k] = float(idata.derived[k])

    print(f"Reference: {REFERENCE_RUN}")
    print(f"{'quantity':<12}{'this run':>16}{'reference':>16}{'rel.diff':>12}   ok")
    print("-" * 62)
    nbad = 0
    for k in ("Cv", "Ci", "C2i", "C3i", "CiL", "CaiL", "CvL", "CavL",
              "CiL_i", "CaiL_i", "CvL_v", "CavL_v",
              "strain_a", "strain_c", "hardening",
              "Z_i_a", "Z_v_a", "Z_i_c", "Z_v_c"):
        if k not in target or k not in got:
            continue
        r = abs(got[k] - target[k]) / max(abs(target[k]), 1e-300)
        ok = r <= TOL_REL
        nbad += (not ok)
        print(f"{k:<12}{got[k]:>16.5e}{target[k]:>16.5e}{r:>12.2e}   "
              f"{'yes' if ok else 'NO'}")
    print("-" * 62)
    if nbad == 0:
        print(f"MATCH: all quantities agree with the reference within {TOL_REL:.0e}")
    else:
        print(f"{nbad} quantity(ies) DIFFER beyond {TOL_REL:.0e} -- 0-D is NOT reproduced")
elif ref is not None:
    print("No 0-D reference run pinned in this checkout -- skipping the regression check.")

In [ ]:
# ── 0-D figure suite (point defects, loops, growth, creep, hardening, ...) ──
if ref is not None:
    viz = plot_results(sim, ref, save_plots=True, output_dir=str(DIR_0D),
                       sim_config=SOLVER_CONFIG)
    # plot_results creates its own timestamped subdirectory; flatten it into 0d/.
    subdirs = [p for p in DIR_0D.iterdir() if p.is_dir()]
    for sd in subdirs:
        for f in sd.iterdir():
            shutil.move(str(f), str(DIR_0D / f.name))
        sd.rmdir()
    pngs = sorted(DIR_0D.glob("*.png"))
    print(f"\n0-D figures written: {len(pngs)}")
    for p in pngs:
        print(f"  {p.name}")

    if SHOW_INLINE:
        from IPython.display import Image, display
        for p in pngs:
            print(f"\n{p.name}")
            display(Image(filename=str(p), width=680))

## 9. Spatial grid and fast-solve backend

A line of `N_POINTS` quadrature points spans $x\in[0,L]$ with a **perfect-sink grain
boundary at $x=0$**. The domain is sized to a few depletion lengths
$\ell=1/\sqrt{Z_N\rho_N}$ so the boundary layer is resolved.

`FAST_SOLVE_BACKEND="auto"` uses MoDELib when it is built *and* a seeded simulation
directory exists, and otherwise falls back to the analytic boundary layer with a clear
banner — the fallback is never silent.

In [ ]:
# ── Spatial grid ────────────────────────────────────────────────────────────
rho0 = float(idata.material_params["rho"])
Z_N  = float(idata.model_params.get("Z_N", 1.1))
ell  = 1.0 / np.sqrt(Z_N * rho0)
L    = DOMAIN_IN_ELL * ell
x    = np.linspace(0.0, L, N_POINTS)
f_depl = 1.0 - np.exp(-x / ell)            # 0 at the GB -> 1 in the interior

print(f"depletion length ell = {ell*1e9:.2f} nm")
print(f"domain L = {L*1e9:.1f} nm over {N_POINTS} points "
      f"(dx = {(x[1]-x[0])*1e9:.2f} nm)")

In [ ]:
# -- Fast-solve backend selection --------------------------------------------
SIM_DIR = OUTPUT_DIR / "modelib_sim"
_seeded = (SIM_DIR / "inputFiles" / "polycrystal.txt").is_file()

from py_utils import modelib_field as mfield

# -- The genuine field channel -----------------------------------------------
# MoDELib3 carries the cluster-dynamics state as a finite-element FIELD: the CD
# block of evl/evl_<N>.txt holds mSize=4 mobile and iSize=8 immobile components
# at every CD node, and ClusterDynamicsFEM::initializeConfiguration reads that
# block back on startup. The same file is therefore the carrier in BOTH
# directions, on the node set the two codes already share.
#
# This replaces the previous handoff, which collapsed the marched state to four
# scalars (one density and one mean radius per family) and pushed them through
# the microstructure-generator inputs aLoopsDensity.txt / frankLoopsDensity.txt.
# Those inputs are uniform by construction, so every bit of spatial structure the
# slow march had produced was discarded on the way out, and the mobile field came
# back through an unverified column map in F/quadrature_*.txt.
FIELD_EVL = Path(globals().get("FIELD_EVL_DIR", "") or (SIM_DIR / "evl"))
FIELD_BRIDGE = None
if (FIELD_EVL / "cdNodes.txt").is_file():
    try:
        FIELD_BRIDGE = mfield.FieldBridge(FIELD_EVL.parent, paths.MODELIB_MATERIAL)
        FIELD_BRIDGE.latest_evl()          # fail early if no evl_<N>.txt is there
    except Exception as _e:
        print(f"[field] unavailable ({_e}) -- falling back to the placeholder")
        FIELD_BRIDGE = None

if FIELD_BRIDGE is not None:
    ACTIVE_BACKEND = "field"
elif FAST_SOLVE_BACKEND == "auto":
    ACTIVE_BACKEND = "modelib" if (MODELIB_OK and _seeded) else "placeholder"
else:
    ACTIVE_BACKEND = FAST_SOLVE_BACKEND

if ACTIVE_BACKEND == "modelib" and not (MODELIB_OK and _seeded):
    raise RuntimeError(
        "FAST_SOLVE_BACKEND='modelib' but "
        f"{'DDomp is not built' if not MODELIB_OK else 'sim_dir is not seeded'}. "
        f"Seed it with seed_sim_dir_from_tutorial({MODELIB_ROOT!s}, {SIM_DIR!s}) "
        "and run its generateInputFiles.py once.")

print(f"fast-solve backend : {ACTIVE_BACKEND}")

if ACTIVE_BACKEND == "field":
    # The march now runs on the CD node set, not on a synthetic 1-D line.
    N_POINTS = FIELD_BRIDGE.n_nodes
    CD_NODES = FIELD_BRIDGE.nodes
    print(FIELD_BRIDGE.describe())
    _cm = FIELD_BRIDGE.read_mobile_field()
    _pos = _cm[:, 0][_cm[:, 0] > 0]
    print(f"\nmarching {N_POINTS} CD nodes; Cv spans "
          f"{_pos.min():.3e} .. {_pos.max():.3e} "
          f"({_pos.max() / _pos.min():.2e} dynamic range)")
elif ACTIVE_BACKEND == "placeholder":
    print(textwrap.dedent("""
        ****************************************************************
        *  PLACEHOLDER fast solve -- NOT a MoDELib FEM result.          *
        *  C_M*(x) = [1 - exp(-x/ell)] * C_M_bulk(dose)                 *
        *  Spatial structure is ANALYTIC; use for pipeline checks only. *
        ****************************************************************"""))
    if not MODELIB_OK:
        print("  reason: DDomp not built (Section 5).")
    elif not _seeded:
        print(f"  reason: {SIM_DIR}/inputFiles/ not seeded.")
    print(f"  reason: no CD field at {FIELD_EVL}/cdNodes.txt")


def fast_solve_field(dose, Q_per_variant=None):
    """Steady mobile field C_M* of shape (N_POINTS, 4) at the given dose."""
    if ACTIVE_BACKEND == "field":
        # Straight from the CD block: MoDELib's four mobile species ARE
        # ZrMicro's [Cv, Ci, C2i, C3i] -- same order, same units, same nodes.
        # No interpolation and no averaging anywhere in this path.
        return FIELD_BRIDGE.read_mobile_field()
    if ACTIVE_BACKEND == "modelib":
        out = FEMSOLVER.solve(Q_per_variant)
        pts, C_M = out["points"], out["C_M"]
        # Reduce mesh coordinates to distance from the boundary plane x=0.
        d = pts[:, 0]
        order = np.argsort(d)
        return np.array([np.interp(x, d[order], C_M[order, j]) for j in range(4)]).T
    return f_depl[:, None] * refmob(dose)[None, :]


## 10. Operator-split dose march

At each step the frozen mobile field is refreshed, injected into every point, and the
immobile ODEs are advanced pointwise (OpenMP batch, one case per quadrature point).
Snapshots are retained at each entry of `DOSE_SNAPSHOTS`.

In [ ]:
# -- Operator-split march -----------------------------------------------------
def q_per_variant_from_state(Y):
    """Volume-averaged per-family (N [m^-3], r [m]).

    Retained ONLY for the legacy scalar handoff (ACTIVE_BACKEND == "modelib"),
    which drives MoDELib's microstructure generator and can accept nothing but
    uniform values. The field backend never calls this -- it writes the whole
    per-node immobile state through modelib_field.FieldBridge instead.
    """
    A = np.mean(np.asarray(Y), axis=0)
    N_a = (A[4] + A[5]) / Omega
    N_c = (A[6] + A[7]) / Omega
    l_a = float(idata.derived.get("l_a", 1.0874628400761981e-10))
    l_c = float(idata.derived.get("l_c", 8.612159651798597e-11))
    r_a = l_a * np.sqrt(max(A[8] + A[9], 0.0) / max(A[4] + A[5], 1e-300))
    r_c = l_c * np.sqrt(max(A[10] + A[11], 0.0) / max(A[6] + A[7], 1e-300))
    return {"a1": (N_a / 3, r_a), "a2": (N_a / 3, r_a),
            "a3": (N_a / 3, r_a), "c": (N_c, r_c)}


def march(Y, t0, t1, nsub=RUN["substeps_per_interval"]):
    """Advance every point over [t0,t1] with the mobile species frozen.

    The substep count is fixed per dose interval rather than derived from a
    wall-clock cap, so it does not blow up when the interval is widened: a 5 dpa
    step at 1e-7 dpa/s is 5e7 s, which a fixed 5e4 s cap would have split into a
    thousand batch subprocesses.

    Each substep is one OpenMP batch subprocess. build_immobile_cases turns on
    the reduced 9x9 implicit block and the exact AD Jacobian, and the solver
    reuses one set of SUNDIALS objects per thread across all cases in the batch.
    """
    nsub = max(1, int(nsub))
    edges = np.linspace(t0, t1, nsub + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        out = run_immobile_step(base_cli, Y, a, b, base_dir=BASE_DIR)
        nfail = sum(o is None for o in out)
        if nfail:
            print(f"     warning: {nfail}/{len(Y)} point(s) failed on "
                  f"[{a:.3e},{b:.3e}] s -- previous state kept")
        Y = [o if o is not None else y for o, y in zip(out, Y)]
    return Y


def push_immobile_field(Y, dose):
    """0-D -> 3-D: write the per-node immobile state into the CD block.

    Writes evl/evl_coupled_<dose>dpa.txt rather than overwriting the source
    configuration, so a march can be repeated without having destroyed its own
    starting point. Hand this file to MoDELib as the restart configuration to
    pass the marched loop population back to the spatial solve.
    """
    if FIELD_BRIDGE is None:
        return None
    dest = FIELD_BRIDGE.evl_dir / f"evl_coupled_{dose:07.3f}dpa.txt"
    return FIELD_BRIDGE.write_immobile_field(np.asarray(Y), dest=dest)


history = {}
field_files = {}
if RUN_3D_MARCH and ref is not None:
    times = DOSE_SNAPSHOTS / G

    im0, rho_seed = refimm(DOSE_SNAPSHOTS[0]), refrho(DOSE_SNAPSHOTS[0])
    CM0 = fast_solve_field(DOSE_SNAPSHOTS[0])

    if ACTIVE_BACKEND == "field":
        # Seed the immobile state from the 3-D field itself, so the march starts
        # from the spatial code's own population rather than from a single 0-D
        # value replicated across the domain.
        imm_field = FIELD_BRIDGE.read_immobile_field()
        Y = [pack_y0(CM0[q], imm_field[q], rho_seed) for q in range(N_POINTS)]
    else:
        Y = [pack_y0(CM0[q], im0.copy(), rho_seed) for q in range(N_POINTS)]

    history[float(DOSE_SNAPSHOTS[0])] = np.array([y.copy() for y in Y])
    print(f"seeded at {DOSE_SNAPSHOTS[0]:.3f} dpa on {N_POINTS} point(s)")

    for n in range(len(DOSE_SNAPSHOTS) - 1):
        if ACTIVE_BACKEND == "field":
            # 3-D -> 0-D: the whole mobile field, node by node.
            CM = fast_solve_field(DOSE_SNAPSHOTS[n])
        else:
            Qv = q_per_variant_from_state(Y)
            CM = fast_solve_field(DOSE_SNAPSHOTS[n], Qv)
        for q in range(N_POINTS):
            Y[q][0:4] = CM[q]
        Y = march(Y, times[n], times[n + 1])
        d1 = float(DOSE_SNAPSHOTS[n + 1])
        history[d1] = np.array([y.copy() for y in Y])

        # 0-D -> 3-D: the whole immobile field, node by node.
        _f = push_immobile_field(history[d1], d1)
        if _f is not None:
            field_files[d1] = str(_f)
        print(f"  marched to {d1:.3f} dpa" + (f"  -> {_f.name}" if _f else ""))

    print(f"\nmarch complete: {len(history)} snapshots")
    if field_files:
        print(f"CD fields written: {len(field_files)} file(s) in "
              f"{FIELD_BRIDGE.evl_dir}")
elif RUN_3D_MARCH:
    print("Cannot march without the 0-D reference (RUN_0D_REFERENCE = False).")
else:
    print("RUN_3D_MARCH = False -- skipping.")


In [ ]:
# ── Regression check against the 0-D reference ──────────────────────────────
if history:
    d_end = max(history)
    final = history[d_end]
    ea_int, ec_int = eps_from_y(final[-1])       # interior point
    ea_ref, ec_ref = eps_from_ref(d_end)         # 0-D reference
    ea_bnd, ec_bnd = eps_from_y(final[0])        # grain boundary

    print(f"At {d_end:.3f} dpa")
    print(f"  interior eps_a = {ea_int:.5e}    0-D eps_a = {ea_ref:.5e}    "
          f"rel.diff = {abs(ea_int-ea_ref)/max(abs(ea_ref),1e-30):.2%}")
    print(f"  interior eps_c = {ec_int:.5e}    0-D eps_c = {ec_ref:.5e}    "
          f"rel.diff = {abs(ec_int-ec_ref)/max(abs(ec_ref),1e-30):.2%}")
    ratio = ea_bnd / max(ea_int, 1e-30)
    kind = "DEPLETED (denuded zone)" if ratio < 1 else "ENHANCED (not a denuded zone)"
    print(f"  boundary eps_a = {ea_bnd:.5e}  = {ratio:.1%} of interior -> {kind}")

    # Diagnose the boundary behaviour: a-loop growth is the small difference of
    # two nearly equal fluxes, so its sign near the GB is delicate. The jump
    # frequencies live on ReactionRates (NOT in input_data.derived).
    d1 = np.argmin(np.abs(ref_dose - d_end))
    fi = float(rr.omega_i * (Cref["Ci"][d1] + 2 * Cref["C2i"][d1] + 3 * Cref["C3i"][d1]))
    fv = float(rr.omega_v * Cref["Cv"][d1])
    Zia = float(idata.derived["Z_i_a"])
    Zva = float(idata.derived["Z_v_a"])
    drive = Zia * fi - Zva * fv
    print(f"  bulk fluxes at {d_end:.2f} dpa: phi_i = {fi:.4e}, phi_v = {fv:.4e}"
          f"   (phi_v/phi_i = {fv/max(fi,1e-300):.4f})")
    print(f"  a-loop drive  Z_i^a*phi_i - Z_v^a*phi_v = {drive:+.4e}"
          f"   = {abs(drive)/max(Zia*fi,1e-300):.1%} of the growth term")
    G_iL_tot = float(idata.derived["G_iL"] + idata.derived["G_aiL"])
    print(f"  mobile-independent cascade deposition over {d_end:.2f} dpa: "
          f"{G_iL_tot * d_end / G:.4e}  (cf. boundary eps_a above)")

    print(textwrap.dedent("""
        NOTE: with the placeholder backend the interior/0-D agreement tests the
        SPLITTING and the pointwise march only. It is not a validation of the
        FEM transport, and -- until the DAD refit -- not a 0-D/3-D physics test.

        WARNING on the boundary value: the placeholder scales ALL FOUR mobile
        species by the SAME factor f(x) = 1 - exp(-x/ell). That preserves the
        ratio phi_v/phi_i while driving both to zero at the GB, so near x = 0 it
        removes loop *absorption* and coalescence while leaving the
        mobile-independent cascade deposition (G_iL, G_aiL) untouched. When the
        a-loop drive printed above is a small difference of two nearly equal
        fluxes, the boundary can therefore end up ENHANCED rather than denuded.
        That is an artifact of the placeholder, not a prediction: a real GB
        depletes interstitials and vacancies over DIFFERENT lengths (their
        diffusivities differ by ~4 orders of magnitude here), which the single
        shared f(x) cannot represent. Do not read the boundary profile
        physically until the MoDELib FEM backend is active."""))

## 11. 3-D field plots

Mobile concentrations, and immobile densities and contents, at every snapshot dose.
Written to `3d/`.

In [ ]:
# ── 3d/ : Figs 19-23, one quantity per file, at each dose interval ──────────
# Fig 19  mobile   Cv, Ci, C2i, C3i
# Fig 20  density  N_c, N_a1, N_a2, N_a3
# Fig 21  content  C_c, C_a1, C_a2, C_a3
# Figs 22-23  per-family platelet overlays on the family's own content field
from py_utils import modelib_report as mreport

_have_evl = (EVL_DIR / "cdNodes.txt").is_file()
if _have_evl:
    _avail = max(int(p.stem.split("_")[1])
                 for p in EVL_DIR.glob("evl_*.txt")) + 1
    DOSES_3D = [d for d in DOSE_SNAPSHOTS if d <= _avail + 1e-9]
    if len(DOSES_3D) < len(DOSE_SNAPSHOTS):
        print(f"NOTE: evl/ holds {_avail} dpa; rendering "
              f"{[float(d) for d in DOSES_3D]} of "
              f"{[float(d) for d in DOSE_SNAPSHOTS]}")
    print(f"3d/ figures from {EVL_DIR}")
    _w = mreport.write_3d_figures(
        EVL_DIR, DOSES_3D, DIR_3D, dose_per_step=DOSE_PER_STEP,
        overlays=FIG3D_OVERLAYS, n_loops=FIG3D_N_LOOPS,
        plane=FIG3D_PLANE)
    print(f"\n{len(_w)} figures written to 3d/")
else:
    print(f"No cluster-dynamics output at {EVL_DIR} -- skipping 3d/.")
    print("Run the 3-D solve (command printed above), then set EVL_DIR to its evl/.")

# ── The finite-element mesh itself ──────────────────────────────────────────
# Drawn from the mesh generated in Section 6a (which is the mesh the sim dir was
# configured with), cut in half so the interior element sizes -- and the
# boundary-layer refinement -- are visible rather than only the surface facets.
_mi = mreport.write_mesh_figure(
    MESH_RESULT.path, DIR_3D / "fe_mesh.png",
    extents_nm=MESH_RESULT.extents_nm, cut="y",
    title=(f"FE mesh: {GEOMETRY['type']}, "
           f"{_lx:.0f}x{_ly:.0f}x{_lz:.0f} nm, "
           f"{MESH_RESULT.n_tets} tets (cut at mid-y)"))
print(f"  fe_mesh.png   {_mi['n_tets']} tets, {_mi['n_shown']} shown, "
      f"{_mi['n_faces']} exposed faces")


## 12. Grain-boundary profiles

Density and content against distance from the boundary, on a **logarithmic $y$ axis**
against linear distance $x$ — the denuded-zone signature. Written to `gb/`.

In [ ]:
# ── gb/ : Figs 24-27, one quantity per file, all doses overlaid ─────────────
# Fig 24  mobile concentrations vs distance from the grain boundary
# Fig 25  loop number density
# Fig 26  stored defect content
# Fig 27  mean loop diameter (from the BINNED mean content and density -- size
#         is a ratio of two fields, and the mean of a ratio is not the ratio of
#         the means)
if _have_evl:
    print(f"gb/ figures from {EVL_DIR}")
    _w = mreport.write_gb_figures(
        EVL_DIR, DOSES_3D, DIR_GB, dose_per_step=DOSE_PER_STEP,
        max_nm=GB_MAX_NM, n_bins=GB_N_BINS)
    print(f"\n{len(_w)} figures written to gb/")
    print(textwrap.dedent("""
        Reading these: adjacent to the boundary there are many small loops, in
        the interior fewer large ones. The stored content is nevertheless
        DEPLETED at the boundary for the <c> family -- its density is nearly
        uniform, so the size deficit is unopposed -- while the <a> families are
        ENHANCED, the density excess outweighing the size deficit.

        The near-boundary loop density is NOT quantitative. Cascade nucleation
        is spatially uniform but the only removal channel is coalescence, driven
        by the absorbed mobile flux, which vanishes where the Dirichlet
        condition pins the mobile species. The boundary is a sink for mobile
        defects but not for loops."""))
else:
    print("No cluster-dynamics output -- skipping gb/.")

# ── Loop size distribution over the cube, at each dose ──────────────────────
# The model carries ONE mean size per family per node, so this is the
# distribution of the LOCAL MEAN diameter across the domain, weighted by the
# local loop density -- not a per-loop size spectrum. Spatial variation is the
# only source of spread, which is exactly what makes it informative here: the
# near-boundary shell contributes small, very numerous loops and the interior
# large, sparse ones, so the distribution is bimodal.
if _have_evl:
    print("size distributions")
    _w = mreport.write_size_distribution_figures(
        EVL_DIR, DOSES_3D, DIR_GB, dose_per_step=DOSE_PER_STEP)
    print(f"\n{len(_w)} size-distribution figures written to gb/")


## 13. Provenance and manifest

Records the environment, the resolved controls, the active backend, and every file
written, so a run is reproducible from its own output.

In [ ]:
# ── Write provenance.md, the raw march data, and a manifest ─────────────────
if history:
    np.savez_compressed(
        RUN_DIR / "march_state.npz",
        x=x, doses=np.array(sorted(history)),
        states=np.array([history[d] for d in sorted(history)]),
        state_names=np.array(["Cv", "Ci", "C2i", "C3i", "CiL", "CaiL", "CvL", "CavL",
                              "CiL_i", "CaiL_i", "CvL_v", "CavL_v"] +
                             [f"acc{i}" for i in range(6)] + ["rho_N"]),
    )
    print(f"  saved march_state.npz  "
          f"({len(history)} doses x {N_POINTS} points x 19 states)")

lines = [
    "# Coupled 0-D / 3-D ZrMicro run", "",
    f"- generated: {datetime.datetime.now().isoformat(timespec='seconds')}",
    f"- repo root: `{REPO_ROOT}`",
    f"- git hash: `{paths.git_hash()}`",
    f"- MoDELib root: `{MODELIB_ROOT}`",
    f"- platform: {platform.platform()}",
    f"- python: {sys.version.split()[0]}",
    f"- input workbook: `{INPUT_FILE.name}`", "",
    "## Controls", "",
    "| control | value |", "|---|---|",
    f"| geometry | {GEOMETRY['type']} |",
    f"| extents | {_lx:.1f} x {_ly:.1f} x {_lz:.1f} nm |",
    f"| mesh | {MESH_RESULT.n_nodes} nodes / {MESH_RESULT.n_tets} tets, order {MESH['element_order']} |",
    f"| element size | {_lc:.2f} nm interior" + (f", {_lc*MESH['boundary_size_ratio']:.2f} nm within {MESH['boundary_thickness_nm']:g} nm of the surface |" if MESH['boundary_layer'] else " (uniform) |"),
    f"| mesh file | `{MESH_RESULT.path.name}` |",
    f"| dose schedule | {DOSE_START:g} -> {DOSE_MAX:g} dpa in {RUN['n_intervals']} intervals of {DOSE_INTERVAL:g} dpa |",
    f"| snapshots | {', '.join(f'{d:g}' for d in DOSE_SNAPSHOTS)} dpa |",
    f"| temperature | {T_K} K |",
    f"| dose rate | {G:.3e} dpa/s |",
    f"| applied stress | {SIGMA_N_PA:.3e} Pa |",
    f"| march grid points | {N_POINTS} |",
    f"| substeps / interval | {RUN['substeps_per_interval']} |",
    f"| rtol / atol | {RUN['rtol']:.1e} / {RUN['atol']:.1e} |",
    f"| 3-D evl source | `{EVL_DIR}` |",
    f"| domain length | {L*1e9:.2f} nm ({DOMAIN_IN_ELL} x ell) |",
    f"| depletion length | {ell*1e9:.2f} nm |",
    f"| fast-solve backend | **{ACTIVE_BACKEND}** |",
    f"| ZrMicro solver | `{ZRMICRO_SOLVER}` |",
    f"| MoDELib DDomp | `{FEMSOLVER.find_ddomp()}` |", "",
]
if ACTIVE_BACKEND == "placeholder":
    lines += ["> **Placeholder fast solve** — spatial structure is analytic,",
              "> not a MoDELib FEM result.", ""]
lines += ["## Known non-correspondences (deliverable Sec. 4.3)", "",
          "1. DAD bias: reconciled by the fitted (Z0_m, p_m) in Zr3d_ghoniem.txt,",
          "   valid only at the fitted temperature -- refit if T changes.",
          "2. Bi-pyramid vacancy family: 3-D only.",
          "3. Size-dependent vacancy-loop emission: 3-D only.", "",
          "## Files", ""]
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        lines.append(f"- `{p.relative_to(RUN_DIR).as_posix()}` "
                     f"({p.stat().st_size/1024:.1f} KB)")

(RUN_DIR / "provenance.md").write_text("\n".join(lines), encoding="utf-8")

nfiles = sum(1 for p in RUN_DIR.rglob("*") if p.is_file())
print(f"\n{'='*66}")
print(f"RUN COMPLETE -> {RUN_DIR}")
print(f"  {nfiles} files   backend = {ACTIVE_BACKEND}")
print(f"  0d/ : {len(list(DIR_0D.glob('*.png')))} figures")
print(f"  3d/ : {len(list(DIR_3D.glob('*.png')))} figures")
print(f"  gb/ : {len(list(DIR_GB.glob('*.png')))} figures")
print(f"{'='*66}")

## 14. Coupled march versus standalone MoDELib3 &mdash; the 1 &micro;m reference

Both routes are advanced from **one shared microstructure**: the 0-D state at
`dose_seed` written uniformly onto the 24&nbsp;115 CD nodes of the 1&nbsp;&micro;m cube,
and both run the **same operator split**.

MoDELib3's `ClusterDynamicsFEM::solve()` *is* the split:

```cpp
solveMobileClusters();                      // FAST: steady C_M*(x). No time
                                            // derivative; the immobile state
                                            // enters through ImmobileSinks
                                            // evaluated at every quad point.
if (useImmobileSolver) solveImmobileClusters();   // SLOW: dt update of n_I, c_I
```

| Route | Fast step | Slow step | Order | Error control |
|---|---|---|---|---|
| standalone | `solveMobileClusters` | MoDELib3 semi-implicit, nodal, 20 substeps/dpa | 1 | none |
| coupled | `solveMobileClusters`, called per substep | CVODE BDF, `acc_mode=2`, exact AD Jacobian | up to 5 | `rtol=1e-6` |

The coupled route calls DDomp with **`useImmobileSolver=0`**, so MoDELib solves
the fast step for the immobile state the march is currently holding and passes
the immobile field straight through; the march then freezes that
$\mathbf C_M^*(x)$ and integrates the immobile ODEs at every node. The mobile
field is therefore re-established against the evolving immobile state at every
substep, which is what makes this an operator split rather than a replay.

**Why that matters at a low-dose seed.** The 0-D seed is spatially uniform and
has no boundary layer, while the true $\mathbf C_M^*(x)$ is pinned to thermal
equilibrium on all six faces &mdash; so the seed is *not* at quasi-steady state.
With the fast step in the loop the march relaxes it on the first call; the table
below reports by how much. `mobile_mode="replay"` reproduces the earlier
arrangement, in which the mobile field came from recorded standalone snapshots
and no such relaxation was possible.

What is left between the routes is **not** the integrator alone: the two
immobile right-hand sides are different models (DAD bias, the bi-pyramid
vacancy family, size-dependent vacancy-loop emission &mdash; see `CLAUDE.md`).

The standalone solve is run separately (a DDomp job of roughly forty minutes for
this schedule); the cell below reads its `evl/` snapshots.

```powershell
# stage the seeded case, then:
wsl.exe -e /mnt/d/GitHub/DisloCluster/MoDELib3/build_dc/tools/DDomp/DDomp `
        /mnt/d/GitHub/DisloCluster/MoDELib3/tutorials/zrmicro_seeded
```

Output follows the reference layout: `0d/`, `3d/`, `gb/`, plus `tables/` with
the comparison and `evl_coupled/` holding the marched fields as restartable
MoDELib configurations. The fast solves run in a **separate** simulation
directory (`<standalone>_qssa`), because each one overwrites `evl/evl_0.txt`.


In [ ]:
# -- Coupled march versus standalone MoDELib3 --------------------------------
from py_utils import run_coupled_vs_standalone as rcs
import importlib; importlib.reload(rcs)

STANDALONE_SIM = paths.MODELIB_ROOT / "tutorials" / "zrmicro_seeded"

# Keep the driver's schedule in step with the RUN block above, so editing one
# control block still governs the whole notebook.
rcs.DOSE_SEED             = RUN["dose_seed"]
rcs.DOSE_MAX              = RUN["dose_max"]
rcs.DOSE_INTERVAL         = (RUN["dose_max"] - RUN["dose_seed"]) / RUN["n_intervals"]
rcs.SUBSTEPS_PER_INTERVAL = RUN["substeps_per_interval"]

# How often the fast step is re-solved, in immobile substeps. 1 is the split as
# written. Each solve is one DDomp invocation, which pays a full FE assembly and
# factorization, so this is the knob that trades splitting accuracy for wall
# clock -- see the fast/slow columns of the timing table.
FEM_EVERY   = 5
MOBILE_MODE = "qssa"      # "qssa" | "replay" (the old arrangement) | "seed"

_ready = (STANDALONE_SIM / "evl" / "cdNodes.txt").is_file()
if not _ready:
    print(f"standalone case not staged at {STANDALONE_SIM}")
    print("run the setup + DDomp steps in the markdown cell above first.")
else:
    dm = rcs.standalone_dose_map(STANDALONE_SIM, G)
    want = [rcs.evl_step_for(d) for d in rcs.dose_snapshots()[1:]]
    missing = [n for n in want if n not in dm]
    print(f"standalone snapshots available: {len(dm)}"
          + (f"; still missing evl_{missing}" if missing else "; complete"))

    # --ddomp-wall lets the speed table quote a measured standalone wall time
    # rather than leaving it blank; pass 0 if the run was not timed.
    OUT = rcs.main([
        "--standalone-sim", str(STANDALONE_SIM),
        "--tag", "coupled_vs_standalone",
        "--mobile-mode", MOBILE_MODE,
        "--fem-every", str(FEM_EVERY),
        "--ddomp-wall", str(globals().get("DDOMP_WALL_S", float("nan"))),
        "--figures",
    ])
    print(f"\n{OUT}")


In [ ]:
# -- Display the comparison tables -------------------------------------------
from IPython.display import Markdown, display

_tbl = OUT / "tables" / "comparison.md" if "OUT" in globals() else None
if _tbl and _tbl.is_file():
    display(Markdown(_tbl.read_text(encoding="utf-8")))
else:
    print("no comparison table yet -- run the cell above")
